In [ ]:
# | default_exp features.mds

In [ ]:
# | export
from __future__ import annotations
import pandas as pd
import structlog
from kreview.eval_engine import FeatureEvaluator

log = structlog.get_logger()

In [ ]:
# | export
class MDSOnTargetEvaluator(FeatureEvaluator):
    """On-target MDS signature.

    Extracts ALL numeric columns from the single-row MDS on-target parquet
    rather than just the 2 originally hardcoded scalars.
    """

    name = "MdsOnTarget"
    source_file = ".MDS.ontarget.parquet"
    tier = 2
    category = "epigenetics_and_geometry"

    def extract(self, df: pd.DataFrame) -> dict[str, float]:
        extracted = {}
        try:
            if df.empty:
                return extracted

            # Extract all numeric columns from the (typically 1-row) MDS parquet
            for col in df.select_dtypes(include="number").columns:
                val = df[col].iloc[0]
                if pd.notna(val):
                    safe_col = str(col).replace(" ", "_").replace("-", "_")
                    extracted[f"mds_ontarget_{safe_col}"] = float(val)

            return extracted
        except Exception as e:
            log.exception("extraction_failed", evaluator=self.name, error=str(e))
            return {}

In [ ]:
# | test
evaluator = MDSEvaluator()
df = pd.DataFrame([{"MDS": 0.5, "mds_ontarget_z": 1.1}])
res = evaluator.extract(df)
assert res["global_mds"] == 0.5